# Global Flight Fare Prediction — Data Preprocessing

This portfolio notebook documents the preprocessing workflow from a collaborative Master of Data Science project. My portfolio focus is data quality, integration across six source tables, transformation, reduction and feature preparation.

**Important:** This notebook is adapted from the original group coursework. The full pipeline requires the public Kaggle dataset listed in the repository README and an internet connection for the initial download. Raw data is not committed to this repository. Review all cells and run from top to bottom in a fresh environment before relying on the outputs.

## Workflow

1. Inspect and clean the six source datasets.
2. Aggregate tables with finer granularity before joining.
3. Integrate ticket, fuel, financial, route and conflict information.
4. Transform and encode variables.
5. Reduce redundant and potentially leakage-prone features.
6. Validate and export the prepared dataset.

---


# 1. Data Preprocessing


## 1.1. Data Cleaning

Import dataset from Kaggle.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("zkskhurram/airline-ticket-prices-vs-oil-and-fuel-costs")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

Import all 6 datasets.

####1.1.1. Datasets Initial Inspection

In [ ]:
financial_df = pd.read_csv(path + '/airline_financial_impact.csv')
ticket_price_df = pd.read_csv(path + '/airline_ticket_prices.csv')
conflicts_df = pd.read_csv(path + '/conflict_oil_events.csv')
fuel_df = pd.read_csv(path + '/fuel_surcharges.csv')
jet_fuel_prices_df = pd.read_csv(path + '/oil_jet_fuel_prices.csv')
route_cost_df = pd.read_csv(path + '/route_cost_impact.csv')

#Check datasets, rows and columns

print('\n1. Financial\n')
display(financial_df.head())

print('\n2. Ticket Prices\n')
display(ticket_price_df.head())

print('\n3. Conflicts\n')
display(conflicts_df.head())

print('\n4. Fuel Surcharges\n')
display(fuel_df.head())

print('\n5. Oil Jet Fuel Prices\n')
display(jet_fuel_prices_df.head())

print('\n6. Route Cost Impact\n')
display(route_cost_df.head())

#Display rows and column of each table


print('\nShapes of each dataset\n')

datasets = {
    "Financial": financial_df,
    "Ticket Prices": ticket_price_df,
    "Conflicts": conflicts_df,
    "Fuel Surcharges": fuel_df,
    "Oil Jet Fuel Prices": jet_fuel_prices_df,
    "Route Cost Impact": route_cost_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")


#Display name of each columns in each table

print('\nColumns of each dataset\n')

for name, df in datasets.items():
    print(f"{name}: {df.columns.tolist()}")

####1.1.2. Descriptive Statistics

In [ ]:
from IPython.display import display, HTML

for name, df in datasets.items():
    display(HTML(f"<h3>{name}</h3>"))

    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

    print("Data types:")
    print(df.dtypes)

    print("\nStatistical summary:")
    display(df.describe(include='all'))

    print("="*80)

####1.1.3. Find Missing Values in Each Tables



In [ ]:
for name, df in datasets.items():
    print(f"\n=== {name} ===")
    missing = df.isnull().sum()
    missing = missing[missing > 0]   # only show columns that actually have missing values
    if missing.empty:
        print("No missing values")
    else:
        print(missing)



1.   yoy_price_change_pct is a column for Year-on-Year percentage change in price ticket that measures the changes from the same month in previous year (e.g. March 2021 to March 2022). The 1980 missing values in this column is not considered a missing data because the entire first year of data (Jan 2019 to Dec 2019) cannot have a YoY value because there is no prior year to compare against. This is structural missingness, not a data error. This goes similar to YoY column in Fuel Surcharges table and Oil Jet Fuel Prices table.



In [ ]:
#Check which row in the Conflict table has missing value.

conflicts_df[conflicts_df['days_since_prev_event'].isnull()]

Since it is the first event and no prior event exists, there is no "correct" numeric value which logically has no "previous event" to count days from. This is structural missingness.

In [ ]:
#Change NaN to 0

conflicts_df['days_since_prev_event'] = conflicts_df['days_since_prev_event'].fillna(0)
conflicts_df[conflicts_df['days_since_prev_event'].isnull()]

####1.1.4. Find Duplicates

In [ ]:
for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    print(f"{name}: {dup_count} duplicate rows")

No duplicates found.

####1.1.5. Correct Inconsistent Data
Important before data integration step.

In [ ]:
#Replacing 'Japanese Airlines' to 'Japan Airlines'

fuel_df['airline'] = fuel_df['airline'].replace('Japanese Airlines', 'Japan Airlines')

####1.1.6. Detect Outliers

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_boxplots(df, name):
    numeric_cols = df.select_dtypes(include='number').columns
    n_cols = len(numeric_cols)

    if n_cols == 0:
        print(f"{name}: no numeric columns")
        return

    # Arrange plots in a grid — 3 columns, as many rows as needed
    n_rows = (n_cols + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

    fig.suptitle(f"Boxplots — {name}", fontsize=14, fontweight='bold', y=1.00)

    for i, col in enumerate(numeric_cols):
        sns.boxplot(x=df[col], ax=axes[i], color='steelblue')
        axes[i].set_title(col, fontsize=10)
        axes[i].set_xlabel('')

    # Hide unused subplots
    for j in range(n_cols, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

for name, df in datasets.items():
    plot_boxplots(df, name)

Visual and contextual inspection revealed that the majority of these flagged observations do not represent data quality errors, but rather legitimate extreme values driven by the underlying phenomena being studied. Therefore, the outliers are retained.

Three factors explain the high outlier count:



1. Event-driven extremes: The dataset spans multiple geopolitical and economic shocks during which variables such as oil prices, passenger volumes, and fare levels recorded values far outside their historical ranges. These observations are central to the research question and must be retained.
2. Heterogeneous carrier groups: The dataset combines flag carriers and low-cost carriers with substantially different fleet sizes, revenues, and cost bases. When analysed in aggregate, this heterogeneity produces apparent outliers that are in fact normal values within each sub-group. Stratifying by airline_type and region resolves most of these cases.
3. Right-skewed distributions: Financial and operational variables such as fleet_size, revenue_usd_m, and passengers_carried_m follow skewed distributions where the IQR assumption of normality does not hold.







####1.1.7. Summary of Data Cleaning


*   Missing values — Most NaNs in year-on-year columns were classified as structural missingness (first 12 months have no prior year) and retained. One NaN in conflict_oil_events was filled with 0 as it referred to the first recorded event.
*   Duplicates — Zero duplicates found across all six datasets.
* Inconsistencies — 'Japanese Airlines' in fuel_surcharges was renamed to 'Japan Airlines' to align with the other datasets.
* Outliers — Box plots flagged many statistical outliers, but contextual inspection confirmed they reflect legitimate extreme values driven by real geopolitical and economic shocks (COVID-19, Ukraine War, US-Iran conflict). Retained rather than removed.

##1.2. Data Integration


In this section, all the 6 datasets will be integrated with the main dataset which is 'Ticket Price' to create one unified dataset. This is essential for data modelling later.

####1.2.1. Join Oil Prices Table to Ticket Price Dataset (Main Dataset)

The first integration step joined oil_jet_fuel_prices to the main ticket price dataset using month as the join key. This dataset provides macro-level oil and fuel market indicators that are common to all airlines in a given month, including Brent crude price, jet fuel price, refinery margin, WTI crude benchmark, OPEC production levels, US strategic reserve levels, and a flag indicating whether the Strait of Hormuz (a critical oil shipping chokepoint) was disrupted. Overall, 7 new columns were integrated).

A left join was used to preserve all 14,355 ticket price rows. Since oil_jet_fuel_prices contains a single row per month (87 rows total), each ticket price row inherits the oil market snapshot of the month in which the ticket was priced. This establishes the macroeconomic backdrop for every observation in the dataset — a necessary foundation for analysing how external oil shocks propagate into ticket pricing.


In [ ]:
# 1. Join oil prices (monthly, global — one row per month)
integrated_df = ticket_price_df.merge(
    jet_fuel_prices_df,
    on='month',
    how='left',
    suffixes=('', '_oil')
)

integrated_df.head()

####1.2.2. Join Airline-Level Fuel Surcharge Dataset
The second integration step joined fuel_surcharges to the dataset using month and airline as composite join keys. This dataset records monthly fuel surcharge amounts broken down by four distance bands (short, medium, long, and ultra-long haul) per airline.

Because this dataset is at a finer granularity than the ticket price data (multiple surcharge bands per airline per month), it was first aggregated to the airline-month level using the mean across surcharge bands. The resulting two columns — average fuel surcharge and average surcharge as a percentage of base fare — were then joined to the main dataset. This provides a complementary view of surcharge behaviour distinct from the route-specific fuel_surcharge_usd column already present in the ticket price dataset.

In [ ]:
# 2. Join fuel surcharges (monthly per airline per band — aggregate first)
fuel_agg = fuel_df.groupby(['month', 'airline']).agg(
    avg_fuel_surcharge_usd = ('fuel_surcharge_usd', 'mean'),
    avg_surcharge_as_pct_base = ('surcharge_as_pct_base', 'mean')
).reset_index()

integrated_df = integrated_df.merge(
    fuel_agg,
    on=['month', 'airline'],
    how='left',
    suffixes=('', '_fuel')
)

####1.2.3. Join Route-Level Cost and Disruption Dataset

The third integration step joined route_cost_impact table to the dataset. This dataset is at the route level (monthly observations of individual routes operated by each airline), making it the most granular source file. Direct joining was therefore not possible as it would have caused row explosion through one-to-many cardinality.

To resolve this, the route data was aggregated by month and airline before joining. Four aggregated metrics were computed: the sum of extra fuel costs from rerouting, the count of rerouted flights, the count of cancelled flights, and the mean fuel percentage of cost across all routes. A left join on month and airline was then used to bring these operational disruption indicators into the main dataset. This enables analysis of how route-level disruption translates into ticket-level price effects.

In [ ]:
# 3. Aggregate route data to airline-month level
route_agg = route_cost_df.groupby(['month', 'airline']).agg(
    total_extra_fuel_cost_usd = ('extra_fuel_cost_usd', 'sum'),
    total_rerouted_flights = ('rerouted', lambda x: (x == 'Yes').sum()),
    total_cancelled_flights = ('flight_cancelled', lambda x: (x == 'Yes').sum()),
    avg_fuel_pct_of_cost = ('fuel_pct_of_cost', 'mean')
).reset_index()

integrated_df = integrated_df.merge(
    route_agg,
    on=['month', 'airline'],
    how='left'
)

####1.2.4. Join Quarterly Financial Data Table (With Temporal Disaggregation)
The fourth integration step joined airline_financial_impact to the dataset using quarter and airline as the composite keys. A quarter column was first derived from the monthly dates in the main dataset to enable this join.

A key methodological challenge arose at this stage due to the granularity mismatch between the two datasets — the financial data is quarterly while the ticket price data is monthly. A naive left join would cause each quarterly value to be replicated across the three constituent months, which is acceptable for ratio variables (such as profit_margin_pct and fuel_hedging_pct) but overstates the magnitude of flow variables (such as passengers_carried_m and hedge_savings_usd_m).

To address this, temporal disaggregation by equal allocation was applied — quarterly flow variables were divided by three to produce monthly estimates before the join. Stock variables (fleet_size) and ratio variables were retained at their native frequency as contextual covariates. This approach preserves the monthly granularity of the analysis while avoiding the overstatement of flow-variable magnitudes at the row level.

Additionally, this join introduces missing values for 8 airlines present in the ticket price dataset but absent from the financial dataset (e.g. ANA, IndiGo, Malaysia Airlines). These rows retain their ticket-level data but are excluded from any analysis requiring financial context.

In [ ]:
# 4. Integrate quarterly financial data via temporal interpolation

# Step A — Add quarter and a midpoint-month anchor to integrated_df and financial_df
integrated_df['quarter'] = (
    pd.to_datetime(integrated_df['month'])
    .dt.to_period('Q').astype(str)
    .str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
)

# Map each quarter to its midpoint month (e.g. 2020-Q1 → 2020-02-01)
quarter_to_midmonth = {'Q1': '02', 'Q2': '05', 'Q3': '08', 'Q4': '11'}

def quarter_to_midpoint_date(q):
    year, qtr = q.split('-')
    return pd.Timestamp(f"{year}-{quarter_to_midmonth[qtr]}-01")

financial_df['anchor_date'] = financial_df['quarter'].apply(quarter_to_midpoint_date)

# Step B — Define which financial columns to interpolate vs hold constant
interpolate_cols = [
    'passengers_carried_m',     # flow — varies smoothly with demand
    'hedge_savings_usd_m',      # flow — varies with market conditions
    'fuel_cost_pct_revenue',    # ratio — varies smoothly with fuel prices
    'profit_margin_pct'         # ratio — varies smoothly with performance
]

constant_cols = [
    'fuel_hedging_pct',         # policy decision — set per quarter
    'fleet_size'                # stock — does not vary smoothly within quarter
]

# Step C — Interpolate per airline across the full monthly date range
all_months = pd.date_range(start='2019-01-01', end='2026-03-01', freq='MS')

interpolated_records = []

for airline, group in financial_df.groupby('airline'):
    group_sorted = group.sort_values('anchor_date').set_index('anchor_date')

    # Interpolate continuous variables
    interp_part = (
        group_sorted[interpolate_cols]
        .reindex(all_months)
        .interpolate(method='linear', limit_direction='both')
    )

    # For flow variables, divide by 3 AFTER interpolation
    # (interpolated value represents the quarterly rate at that month)
    interp_part['passengers_monthly_est'] = interp_part['passengers_carried_m'] / 3
    interp_part['hedge_savings_monthly_est'] = interp_part['hedge_savings_usd_m'] / 3

    # Hold constant variables flat within each quarter (forward-fill from anchor)
    constant_part = (
        group_sorted[constant_cols]
        .reindex(all_months)
        .ffill()
        .bfill()
    )

    combined = pd.concat([interp_part, constant_part], axis=1)
    combined['airline'] = airline
    combined['month'] = combined.index.strftime('%Y-%m')
    interpolated_records.append(combined.reset_index(drop=True))

financial_monthly = pd.concat(interpolated_records, ignore_index=True)

# Step D — Select the columns to merge into integrated_df
financial_cols = [
    'month', 'airline',
    'fuel_cost_pct_revenue',
    'profit_margin_pct',
    'fuel_hedging_pct',
    'hedge_savings_monthly_est',
    'passengers_monthly_est',
    'fleet_size'
]

# Step E — Merge on month + airline (now both monthly)
integrated_df = integrated_df.merge(
    financial_monthly[financial_cols],
    on=['month', 'airline'],
    how='left'
)

print(f"Shape after Round 4: {integrated_df.shape}")
print(f"\nSample interpolated values for Emirates 2020-Q1:")
print(integrated_df[
    (integrated_df['airline']=='Emirates') &
    (integrated_df['month'].isin(['2020-01','2020-02','2020-03']))
][['month','airline','passengers_monthly_est','hedge_savings_monthly_est',
   'profit_margin_pct','fleet_size']].drop_duplicates())

In [ ]:
import matplotlib.pyplot as plt

# Pick one airline to visualise the interpolation effect
airline_to_plot = 'Emirates'

# Get the original quarterly anchored values
original = financial_df[financial_df['airline'] == airline_to_plot].copy()
original_dates = original['anchor_date']
original_passengers = original['passengers_carried_m'] / 3   # convert to monthly equivalent
original_profit = original['profit_margin_pct']

# Get the interpolated monthly values
interp = integrated_df[integrated_df['airline'] == airline_to_plot][
    ['month', 'passengers_monthly_est', 'profit_margin_pct']
].drop_duplicates().sort_values('month')

interp_dates = pd.to_datetime(interp['month'])

# Plot two side-by-side charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Chart 1: Passengers (flow variable) ---
axes[0].plot(interp_dates, interp['passengers_monthly_est'],
             color='steelblue', linewidth=2, label='Interpolated (monthly)')
axes[0].scatter(original_dates, original_passengers,
                color='red', s=60, zorder=5, label='Original quarterly anchor')
axes[0].set_title(f'{airline_to_plot} — Passengers (Monthly Estimate)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Passengers (millions, monthly)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- Chart 2: Profit margin (ratio variable) ---
axes[1].plot(interp_dates, interp['profit_margin_pct'],
             color='darkgreen', linewidth=2, label='Interpolated (monthly)')
axes[1].scatter(original_dates, original_profit,
                color='red', s=60, zorder=5, label='Original quarterly anchor')
axes[1].set_title(f'{airline_to_plot} — Profit Margin (%)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Profit margin (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Verify Emirates Q1 2020 values are now different across months
# Works whether month is string or datetime
month_filter = pd.to_datetime(['2020-01', '2020-02', '2020-03'])

sample = integrated_df[
    (integrated_df['airline'] == 'Emirates') &
    (pd.to_datetime(integrated_df['month']).isin(month_filter))
][['month', 'passengers_monthly_est', 'profit_margin_pct',
   'fuel_hedging_pct', 'fleet_size']].drop_duplicates().sort_values('month')

print(sample)

####1.2.5. Join Conflict and Oil Event Data
The final integration step joined conflict_oil_events to the dataset using conflict_phase as the join key. This dataset is event-level rather than time-series — it records 36 distinct geopolitical and supply events across the study period, each tagged with one of eight conflict phases.
Because conflict events cannot be joined directly to each row (they do not align with specific airlines or months), the event data was aggregated by conflict phase to produce a phase-level summary. Three metrics were computed: the count of events per phase, the mean severity score (with severity levels mapped to an ordinal 1–4 scale), and the sum of estimated airfare impacts.
These three columns were then joined onto the main dataset by conflict_phase. The result is that every row now carries the broader event context of the period it belongs to — for example, observations during the US-Iran War Conflict phase are tagged with 9 events at an average severity of 3.33 and a cumulative estimated airfare impact of +153.6%. This provides an aggregate measure of the geopolitical intensity that serves as a contextual variable for the passenger pricing analysis.

In [ ]:
# 5. Aggregate events by phase (count + avg severity impact)
severity_map = {'Medium': 1, 'High': 2, 'Very High': 3, 'Extreme': 4}
conflicts_df['severity_score'] = conflicts_df['severity'].map(severity_map)

event_agg = conflicts_df.groupby('conflict_phase').agg(
    event_count = ('event_date', 'count'),
    avg_severity = ('severity_score', 'mean'),
    total_airfare_impact_pct = ('airfare_impact_pct', 'sum')
).reset_index()

integrated_df = integrated_df.merge(
    event_agg,
    on='conflict_phase',
    how='left'
)

In [ ]:
print(f"Final shape: {integrated_df.shape}")
print(f"Original ticket_price_df shape: {ticket_price_df.shape}")

# Confirm row count hasn't exploded or shrunk
assert integrated_df.shape[0] == ticket_price_df.shape[0], "Row count changed — check join keys!"

# Check NaN coverage
print("\nMissing values per column:")
print(integrated_df.isnull().sum()[integrated_df.isnull().sum() > 0])

integrated_df.head()

####1.2.6. Handle Missing Values Post-Integration


Following integration, missing values introduced by unmatched rows in the fuel surcharges and route cost datasets were resolved — route disruption columns were filled with 0 (no disruption occurred) and surcharge percentage was imputed using each airline's median. Year-on-year change columns retain structural NaNs in the first 12 months as documented in the cleaning phase.

In [ ]:
# === Check missing values after integration (before fixing) ===

print("=" * 60)
print("MISSING VALUES CHECK — POST INTEGRATION")
print("=" * 60)

missing = integrated_df.isnull().sum()
missing = missing[missing > 0]

if missing.empty:
    print("\n✓ No missing values found.")
else:
    print(f"\nTotal columns with missing values: {len(missing)}")
    print(f"Total missing cells: {missing.sum():,}")
    print()
    print(f"{'Column':<35} {'Missing':>8} {'% of rows':>10}")
    print("-" * 55)
    for col, count in missing.items():
        pct = count / len(integrated_df) * 100
        print(f"  {col:<33} {count:>8,} {pct:>9.2f}%")

In [ ]:
# === Handle missing values post-integration ===

# 1. Route disruption columns — NaN means no disruption, fill with 0
route_cols = [
    'total_extra_fuel_cost_usd',
    'total_rerouted_flights',
    'total_cancelled_flights',
    'avg_fuel_pct_of_cost'
]
integrated_df[route_cols] = integrated_df[route_cols].fillna(0)
print(f"Route disruption NaNs after fix: {integrated_df[route_cols].isnull().sum().sum()}")

# 2. avg_surcharge_as_pct_base — fill with airline-level median
integrated_df['avg_surcharge_as_pct_base'] = integrated_df.groupby('airline')['avg_surcharge_as_pct_base'].transform(
    lambda x: x.fillna(x.median())
)
print(f"avg_surcharge_as_pct_base NaNs after fix: {integrated_df['avg_surcharge_as_pct_base'].isnull().sum()}")

# 2b. Fallback — airlines with no surcharge data at all get the global median
integrated_df['avg_surcharge_as_pct_base'] = integrated_df['avg_surcharge_as_pct_base'].fillna(integrated_df['avg_surcharge_as_pct_base'].median()
)
print(f"avg_surcharge_as_pct_base NaNs after global fallback: {integrated_df['avg_surcharge_as_pct_base'].isnull().sum()}")

# 3. yoy columns — retain as structural missingness (first 12 months have no prior year)
print(f"\nyoy_price_change_pct NaNs: {integrated_df['yoy_price_change_pct'].isnull().sum()} — retained (structural)")
print(f"yoy_brent_change_pct NaNs: {integrated_df['yoy_brent_change_pct'].isnull().sum()} — retained (structural)")

# 4. Final check
print(f"\nRemaining NaNs after all fixes:")
remaining = integrated_df.isnull().sum()
remaining = remaining[remaining > 0]
print(remaining if not remaining.empty else "  ✓ No missing values (excluding structural YoY columns)")

####1.2.7. Summary of Data Integration

*   Five integration rounds combined the six datasets, anchored on airline_ticket_prices.
*   Aggregation before joining was applied to fuel surcharges and route cost data to resolve granularity differences.
*   Temporal interpolation was used (instead of ÷3 disaggregation) for the quarterly financial data, anchoring values at quarter midpoints and linearly interpolating monthly estimates.
*  Conflict event data was aggregated by phase to produce contextual covariates.
*   Post-integration NaNs were resolved — disruption columns filled with 0, surcharge filled with airline median, and structural YoY NaNs retained.
*   All 14,355 ticket-level rows preserved with 26 new contextual columns added.



After all five integration rounds, the final dataset contains 14,355 rows and 45 columns, with the row count unchanged from the original ticket price dataset. The 26 new columns added across the five rounds provide macroeconomic context (oil prices), airline-level context (fuel surcharges and financial performance), operational context (route disruption), and geopolitical context (conflict events) for each ticket price observation. This enables a multi-dimensional analysis of how external shocks translate into the fares experienced by passengers.

##1.3. Data Transformation

####1.3.1. Convert month to datetime

In [ ]:
integrated_df['month'] = pd.to_datetime(integrated_df['month'], format='%Y-%m')

####1.3.2. Convert strait_hormuz_disrupted to binary

The strait_hormuz_disrupted column from oil_jet_fuel_prices was stored as a string with values "Yes" or "No". Because the column is categorical with only two levels, it was converted to binary numeric form (1 for Yes, 0 for No). This transformation reduces memory usage, enables direct use of the column in correlation and regression analyses, and aligns it with the other numeric binary indicators already present in the dataset.

In [ ]:
integrated_df['strait_hormuz_disrupted'] = (
    integrated_df['strait_hormuz_disrupted'].map({'Yes': 1, 'No': 0})
)

The month column was originally stored as a string in YYYY-MM format. It was converted to the pandas datetime type to enable proper time-series operations such as chronological sorting, resampling, and date-based filtering. This is essential for temporal analysis of how ticket prices evolve across months and years.

####1.3.3. Ordinal encoding for conflict_phase

Because the eight conflict phases follow a natural chronological order (from Pre-Pandemic Baseline through to US-Iran War Conflict), ordinal encoding was applied to each phase was mapped to an integer from 1 to 8 reflecting its sequence in time. A new column conflict_phase_ordinal was created, while the original conflict_phase column was retained for grouped visualisations and reporting. This encoding preserves the inherent ordering of the variable while allowing it to be used numerically in regression models.

In [ ]:
phase_order = {
    'Pre-Pandemic Baseline': 1,
    'COVID-19 Collapse': 2,
    'Recovery & Surge': 3,
    'Ukraine War Shock': 4,
    'Stabilisation': 5,
    'Gaza-Israel Conflict': 6,
    'Pre-Iran Escalation': 7,
    'US-Iran War Conflict': 8
}

integrated_df['conflict_phase_ordinal'] = integrated_df['conflict_phase'].map(phase_order)

The ordinal assignment reflects increasing flight distance:


*  1: Short-Haul Domestic - Shortest — within the same country
* 2: Short-Haul Regional - Slightly longer — between nearby countries
* 3: Medium-Haul - Intermediate distance
* 4: Long-Haul - International long distance
* 5: Ultra-Long-Haul - Longest routes






In [ ]:
# Updated ordinal encoding for route_class
route_order = {
    'Short-Haul Domestic': 1,
    'Short-Haul Regional': 2,
    'Medium-Haul': 3,
    'Long-Haul': 4,
    'Ultra-Long-Haul': 5
}
integrated_df['route_class_ordinal'] = integrated_df['route_class'].map(route_order)

# Verify
print(f"route_class_ordinal NaNs: {integrated_df['route_class_ordinal'].isnull().sum()}")
print(f"route_class_ordinal unique values: {sorted(integrated_df['route_class_ordinal'].dropna().unique())}")

####1.3.4. One-hot encoding for route_class and region



In [ ]:
# One-hot encoding for airline_type (only if not already encoded)
if 'airline_type' in integrated_df.columns:
    integrated_df = pd.get_dummies(
        integrated_df,
        columns=['airline_type'],
        drop_first=True,
        dtype=int
    )
    print("airline_type encoded successfully.")
else:
    print("airline_type already encoded — skipping.")

# One-hot encoding for region (only if not already encoded)
if 'region' in integrated_df.columns:
    integrated_df = pd.get_dummies(
        integrated_df,
        columns=['region'],
        drop_first=True,
        dtype=int
    )
    print("region encoded successfully.")
else:
    print("region already encoded — skipping.")

# Verify
print(f"airline_type_Low Cost exists: {'airline_type_Low Cost' in integrated_df.columns}")
print(f"region exists: {'region' in integrated_df.columns}")

####1.3.5. One-hot encoding for airline and country (Changed from: Label encoding for airline and country)

In [ ]:
# airline and country will be one-hot encoded after data reduction (Section 1.4)
# Keeping them as-is here so downstream cells can still reference them by name
print("Skipped — airline and country encoding deferred to post-reduction step.")

####1.3.6. Verify Transformation

In [ ]:
print(f"Shape after transformation: {integrated_df.shape}")

print(f"\nData types summary:")
print(integrated_df.dtypes.value_counts())

print(f"\nNew columns created:")
new_cols = [
    'strait_hormuz_disrupted',
    'conflict_phase_ordinal',
    'route_class_ordinal',
    'airline_type_Low Cost'
]
print(integrated_df[new_cols].head())

# Check fuel_cost_pct_opex still exists
print(f"\nfuel_cost_pct_opex exists: {'fuel_cost_pct_opex' in integrated_df.columns}")

In [ ]:
# Check fuel_cost_pct_opex still exists after all operations
print(f"fuel_cost_pct_opex exists: {'fuel_cost_pct_opex' in integrated_df.columns}")

###1.3.7. Summary of Data Transformation

*   Boolean conversion — strait_hormuz_disrupted mapped to 1/0.
*   Datetime conversion — month converted from string to datetime.
* Ordinal encoding — conflict_phase (1–8) and route_class (1–5), preserving natural order.
*  One-hot encoding — airline_type encoded with drop_first=True to avoid the dummy variable trap.
*  Feature engineering, Scaling and PCA deferred to the ML phase to prevent data leakage during train-test split.










Deferred techniques: Scaling, normalisation, and log transformation were not applied at this stage. These transformations are model-specific and will be applied during the machine learning pipeline phase, after the train/test split is performed, to avoid data leakage and to ensure each model receives appropriately prepared inputs.

##1.4. Data Reduction

####1.4.1. Drop Columns

Data reduction was performed to eliminate redundant, uninformative, and analytically problematic columns following the integration and transformation stages. The goal is to produce a leaner, more focused dataset without sacrificing analytical value.

Three categories of columns were removed:

**Redundant columns** — `jet_fuel_usd_barrel_oil` and `conflict_phase_oil` were duplicates created by the Round 1 merge (suffixed with `_oil`), containing identical information to their non-suffixed counterparts. `data_source` was removed as it carried the same literal string value across every row, offering zero analytical value.

**Target leakage columns** — `base_fare_usd`, `fuel_surcharge_usd`, `taxes_fees_usd`, `fuel_surcharge_pct_of_fare`, and `base_fare_pct_of_total` are direct arithmetic components or derivations of the target variable `total_fare_usd`. Including them as features would allow the model to trivially reconstruct the target, producing artificially perfect accuracy with no real predictive learning.

**Similar info columns** — Four columns were confirmed as highly redundant / bring similar information with existing features through correlation analysis (r > 0.85): `wti_crude_usd_barrel` (r = 0.9981 with `brent_crude_usd`), `jet_fuel_usd_per_gallon` (r = 0.9949 with `jet_fuel_usd_barrel`), `refinery_margin_usd` (r = 0.8926 with `jet_fuel_usd_barrel`), and `avg_fuel_surcharge_usd` (r = 0.8955 with `jet_fuel_usd_barrel`). Retaining these would add noise without adding information, and would cause unstable coefficients in linear models.

In [ ]:
# CATEGORY 1: Drop redundant and zero-information columns
cols_redundant = [
    'jet_fuel_usd_barrel_oil',    # duplicate after Round 1 merge
    'conflict_phase_oil',         # duplicate after Round 1 merge
    'data_source'                 # zero variance — same value every row
]

integrated_df = integrated_df.drop(columns=cols_redundant, errors='ignore')
print(f"Redundant columns dropped: {cols_redundant}")
print(f"Shape after redundant drop: {integrated_df.shape}")

# CATEGORY 2: Drop target leakage columns
cols_leakage = [
    'base_fare_usd',              # direct component: base + surcharge + tax = total_fare_usd
    'fuel_surcharge_usd',         # direct component
    'taxes_fees_usd',             # direct component
    'fuel_surcharge_pct_of_fare', # derived from fuel_surcharge_usd / total_fare_usd
    'base_fare_pct_of_total'      # derived from base_fare_usd / total_fare_usd
]

integrated_df = integrated_df.drop(columns=cols_leakage, errors='ignore')
print(f"\nTarget leakage columns dropped: {cols_leakage}")
print(f"Shape after leakage drop: {integrated_df.shape}")

# CATEGORY 3: Drop columns with similar information
cols_multicollinear = [
    'wti_crude_usd_barrel',
    'jet_fuel_usd_per_gallon',
    'jet_fuel_usd_barrel',
    'refinery_margin_usd',
    'avg_fuel_surcharge_usd',
    'conflict_phase', # encoded to conflict_phase_ordinal
    'route_class', # encoded to route_class_ordinal
    'iata_code' # causes redundancy as it is the identifier for airline name
]

integrated_df = integrated_df.drop(columns=cols_multicollinear, errors='ignore')
print(f"\nSimilar info columns dropped: {cols_multicollinear}")
print(f"Shape after similar info columns drop: {integrated_df.shape}")

# CATEGORY 4: Drop YoY columns due to structural NaNs
cols_yoy = [
    'yoy_price_change_pct',   # 1,500 structural NaNs (first year)
    'yoy_brent_change_pct',   # 1,500 structural NaNs (first year)
]

integrated_df = integrated_df.drop(columns=cols_yoy, errors='ignore')
print(f"\nYoY columns dropped: {cols_yoy}")
print(f"Shape after YoY drop: {integrated_df.shape}")

print(f"\nTotal columns removed: {len(cols_redundant) + len(cols_leakage) + len(cols_multicollinear) + len(cols_yoy)}")
print(f"Final shape after all column drops: {integrated_df.shape}")

####1.4.2. Drop Airlines Without Financial Context After Merging Tables

In [ ]:
# Identify airlines with no financial context (8 airlines absent from financial_df)
airlines_without_financial = integrated_df[
    integrated_df['fleet_size'].isnull()
]['airline'].unique()

print(f"Airlines being removed ({len(airlines_without_financial)}):")
for a in airlines_without_financial:
    print(f"  - {a}")

# Track row count before removal
rows_before = integrated_df.shape[0]

# Remove rows for those airlines
integrated_df = integrated_df[
    ~integrated_df['airline'].isin(airlines_without_financial)
].reset_index(drop=True)

# Report results
rows_after = integrated_df.shape[0]
rows_removed = rows_before - rows_after

print(f"\nRows before removal: {rows_before:,}")
print(f"Rows after removal:  {rows_after:,}")
print(f"Rows removed:        {rows_removed:,} ({rows_removed/rows_before*100:.2f}%)")
print(f"Remaining airlines:  {integrated_df['airline'].nunique()}")
print(f"Final shape:         {integrated_df.shape}")

####1.4.3. Reorder Columns

In [ ]:
new_order = [
    # Group 1 — Temporal Identifiers
    'month', 'quarter', 'conflict_phase_ordinal',

    # Group 2 — Airline Identifiers
    'airline', 'country', 'region', 'airline_type_Low Cost', 'fleet_size',

    # Group 3 — Route Characteristics
    'route_class_ordinal', 'avg_route_km',

    # Group 4 — Ticket Pricing (target)
    'total_fare_usd',
    'yoy_price_change_pct', 'load_factor_pct',

    # Group 5 — Oil & Fuel Market Context
    'brent_crude_usd', 'jet_fuel_usd_barrel',
    'opec_production_mbd', 'us_strategic_reserve_mbl',
    'strait_hormuz_disrupted', 'yoy_brent_change_pct',

    # Group 6 — Airline Financial Context
    'fuel_cost_pct_opex', 'fuel_cost_pct_revenue', 'profit_margin_pct',
    'fuel_hedging_pct', 'hedge_savings_monthly_est',
    'passengers_monthly_est',

    # Group 7 — Disruption & Event Context
    'avg_surcharge_as_pct_base',
    'total_extra_fuel_cost_usd', 'total_rerouted_flights',
    'total_cancelled_flights', 'avg_fuel_pct_of_cost',
    'event_count', 'avg_severity', 'total_airfare_impact_pct'
]

existing_cols = [col for col in new_order if col in integrated_df.columns]
leftover_cols = [col for col in integrated_df.columns if col not in existing_cols]

# Combine them
integrated_df = integrated_df[existing_cols + leftover_cols]

print(f"Reordered dataset shape: {integrated_df.shape}")
print(f"Columns: {list(integrated_df.columns)}")

In [ ]:
print(f"Final columns in integrated_df:")
for i, col in enumerate(integrated_df.columns, 1):
    print(f"  {i}. {col}")

####1.4.4. Renaming Columns for Clarity

In [ ]:
# Rename columns for clarity
column_renames = {
    'month': 'obs_month',
    'quarter': 'obs_quarter',
    'conflict_phase_ordinal': 'conflict_phase_id',

    'airline': 'airline_name',
    'country': 'airline_country',
    'region': 'airline_region',
    'airline_type_Low Cost': 'is_low_cost_carrier',
    'fleet_size': 'airline_fleet_size',

    'route_class_ordinal': 'route_class_id',
    'avg_route_km': 'route_distance_km',

    'total_fare_usd': 'fare_total_usd',
    'load_factor_pct': 'flight_load_factor_pct',

    'brent_crude_usd': 'oil_brent_usd_per_barrel',
    'opec_production_mbd': 'opec_production_mil_bpd',
    'us_strategic_reserve_mbl': 'us_oil_reserve_mil_barrels',
    'strait_hormuz_disrupted': 'is_hormuz_disrupted',

    'fuel_cost_pct_opex': 'fuel_pct_of_operating_cost',
    'fuel_cost_pct_revenue': 'fuel_pct_of_revenue',
    'profit_margin_pct': 'airline_profit_margin_pct',
    'fuel_hedging_pct': 'airline_hedging_coverage_pct',
    'hedge_savings_monthly_est': 'hedge_savings_usd_monthly',
    'passengers_monthly_est': 'passengers_mil_monthly',

    'avg_surcharge_as_pct_base': 'surcharge_pct_of_base_fare',
    'total_extra_fuel_cost_usd': 'rerouting_fuel_cost_usd',
    'total_rerouted_flights': 'rerouted_flights_count',
    'total_cancelled_flights': 'cancelled_flights_count',
    'avg_fuel_pct_of_cost': 'route_fuel_pct_of_cost',
    'event_count': 'phase_event_count',
    'avg_severity': 'phase_avg_severity',
    'total_airfare_impact_pct': 'phase_airfare_impact_pct'
}

integrated_df = integrated_df.rename(columns=column_renames)
print(f"Renamed {len(column_renames)} columns.")
print(f"Final columns:\n{list(integrated_df.columns)}")

In [ ]:
# One-hot encode airline_name and airline_country (nominal categories with no natural order)
integrated_df = pd.get_dummies(integrated_df, columns=['airline_name', 'airline_country'], drop_first=True, dtype=int)

airline_cols = [c for c in integrated_df.columns if c.startswith('airline_name_')]
country_cols = [c for c in integrated_df.columns if c.startswith('airline_country_')]

print(f"Shape after one-hot encoding: {integrated_df.shape}")
print(f"Airline dummy columns created: {len(airline_cols)}")
print(f"Country dummy columns created: {len(country_cols)}")

####1.4.5 Summary of Data Reduction

*   Redundant columns dropped — jet_fuel_usd_barrel_oil and conflict_phase_oil (duplicates created by the Round 1 merge).
*   Zero-variance column dropped — data_source (same value for every row, no analytical value).
*   Rows removed — 8 airlines without financial context (~24% of rows) were excluded to ensure every observation is analysable across both ticket-level and financial dimensions.
*   Original categorical columns retained alongside their encoded versions for visualisations and modelling.
Advanced techniques deferred — Feature selection and PCA will be applied during the machine learning phase, after the train/test split, to ensure model-specific optimisation and prevent data leakage.

The final dataset contains approximately 10,875 rows × 36 columns spanning 25 airlines, ready for exploratory analysis and machine learning modelling.





 ## 1.5 Feature Engineering

*   Extract the Year and Month from absolute_month column
*   Drop the absolute_quarter column Because a quarter is entirely dependent on the month (e.g., January to March is always Q1), keeping the obs_quarter column introduces multicollinearity. This can confuse some models, hence it is best to simply drop it.
*   Apply Cyclical Encoding to help the model understand cyclical nature. For example, naturally the model will think January (encode as 1) and December (encode as 12) are numerically far apart. In reality, it is a nature cycle where December 31st is right next to January 1st. Sine and cosine transformations will be used to help the model understand this cyclical nature:
$month\_sin = \sin(2\pi \cdot \frac{month}{12})$
$month\_cos = \cos(2\pi \cdot \frac{month}{12})$



In [ ]:
# 1. Convert obs_month to actual datetime objects
integrated_df['obs_month'] = pd.to_datetime(integrated_df['obs_month'])

# 2. Extract numerical Year and Month into new columns
integrated_df['year'] = integrated_df['obs_month'].dt.year
integrated_df['month'] = integrated_df['obs_month'].dt.month

# 3. Create Cyclical Features for the month (Highly recommended for seasonality)
integrated_df['month_sin'] = np.sin(2 * np.pi * integrated_df['month'] / 12)
integrated_df['month_cos'] = np.cos(2 * np.pi * integrated_df['month'] / 12)

# 4. Drop the original problematic columns
# We drop 'obs_month' because we extracted its data.
# We drop 'obs_quarter' because it's redundant to 'month'.
integrated_df = integrated_df.drop(columns=['obs_month', 'obs_quarter'])

# Optional: Drop the standard 'month' column if you strictly use the sin/cos versions
# integrated_df = integrated_df.drop(columns=['month'])

In [ ]:
# === FINAL DATASET OVERVIEW ===

print("=" * 70)
print("FINAL INTEGRATED DATASET")
print("=" * 70)

print(f"\nShape: {integrated_df.shape[0]:,} rows × {integrated_df.shape[1]} columns")
print(f"Airlines: {len([c for c in integrated_df.columns if c.startswith('airline_name_')])}")
min_year = integrated_df['year'].min()
min_month = integrated_df[integrated_df['year'] == min_year]['month'].min()
max_year = integrated_df['year'].max()
max_month = integrated_df[integrated_df['year'] == max_year]['month'].max()

print(f"Time range: {min_year}-{min_month:02d} to {max_year}-{max_month:02d}")
#print(f"Time range: {integrated_df['obs_month'].min()} to {integrated_df['obs_month'].max()}")

print(f"\nMissing values check:")
missing = integrated_df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("  ✓ No missing values")
else:
    print(missing)

print(f"\nFirst 5 rows:")
display(integrated_df.head())

print(f"\nLast 5 rows:")
display(integrated_df.tail())

print(f"\nDescriptive statistics:")
display(integrated_df.describe(include='all'))

In [ ]:
# Save the final preprocessed dataset to CSV
integrated_df.to_csv('preprocessed_airline_dataset.csv', index=False)

print(f"Saved! Shape: {integrated_df.shape}")
print(f"File: preprocessed_airline_dataset.csv")

## Portfolio validation notes

Before publishing final outputs, verify the resulting row/column counts against the current source data and this notebook's transformations. The accompanying group report contains multiple dataset-size descriptions from different pipeline stages; use the actual executed output as the source of truth.

For production use, consider moving imputers and encoders into a scikit-learn `Pipeline` fitted on training data only, and evaluate with a time-based split. These are recommended future improvements, not claims about the original coursework implementation.
